#  Generación de Ejemplos Sintéticos 

**Objetivo:** Generar 2,801 ejemplos sintéticos de bullying/hate speech para balancear el dataset

**Método:** OpenAI API (gpt-4o-mini) en lotes de 100 ejemplos

**Dataset actual:**
- Clase 0: 4,366 ejemplos (reales limpios)
- Clase 1: 1,565 ejemplos (reales limpios)
- Ratio: 2.79:1 (desbalanceado)

**Dataset objetivo:**
- Clase 0: 4,366 ejemplos (reales limpios)
- Clase 1: 4,366 ejemplos (1,565 reales + 2,801 sintéticos)
- TOTAL: 8,732 ejemplos
- Ratio: 1.00:1 (balanceado perfectamente)

---

In [1]:
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os
from openai import OpenAI
import json
import time
from pathlib import Path

# Cargar variables del archivo .env
load_dotenv()

# Leer la key
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

print("✅ Cliente OpenAI configurado")

✅ Cliente OpenAI configurado


In [2]:
# Cargar dataset limpio para analizar ejemplos reales
df_limpio = pd.read_csv('../data/processed/zenodo_limpio_textos.csv')

print("="*70)
print("ANALISIS DE EJEMPLOS REALES")
print("="*70)

# Filtrar solo clase 1 (ofensivos)
ejemplos_reales = df_limpio[df_limpio['label'] == 1]['texto'].tolist()

print(f"\nEjemplos reales de clase 1: {len(ejemplos_reales)}")

# Analizar caracteristicas
longitudes = [len(texto) for texto in ejemplos_reales]
palabras = [len(texto.split()) for texto in ejemplos_reales]

print(f"\nCaracteristicas de ejemplos reales:")
print(f"   Longitud promedio: {int(np.mean(longitudes))} caracteres")
print(f"   Palabras promedio: {round(np.mean(palabras), 1)} palabras")
print(f"   Rango: {min(longitudes)}-{max(longitudes)} caracteres")

# Tomar muestra para mostrar a ChatGPT como referencia
muestra_referencia = np.random.choice(ejemplos_reales, 15, replace=False)

print(f"\nMuestra de ejemplos reales (para referencia):")
for i, ejemplo in enumerate(muestra_referencia, 1):
    print(f"   {i}. {ejemplo[:100]}...")

ANALISIS DE EJEMPLOS REALES

Ejemplos reales de clase 1: 1565

Caracteristicas de ejemplos reales:
   Longitud promedio: 88 caracteres
   Palabras promedio: 15.7 palabras
   Rango: 15-280 caracteres

Muestra de ejemplos reales (para referencia):
   1. que por culo me dio el subnormal este, es asqueroso...
   2. Lo de antena3 confirma la teoría d q el votante fascista es inculto,inutil y centrista...
   3. -Emo mongolo con gorra -Dice su edad con lvl -Bio de mongolo. - se patina guai en Sevilla - Dinosaur...
   4. algún problema? -sí cual? -eres maricón...
   5. Ok diario noticias? Facha se te ve el plumero franquista y asesino que tenéis en vena todos, queréis...
   6. cuando todo lo que has estado retuiteando hoy es de cuando ya te habias ido al haber hecho el puto s...
   7. A su vez que no es lo mismo ser cristiano que catolico , pero tu eres CATETA....
   8. Si en vez de Boyé, la película estuviera protagonizada por uno de los fascistas asesinos de los abog...
   9. Un tio que pone

In [3]:
# Configuracion
TOTAL_SINTETICOS = 2801
LOTE_SIZE = 100
MODELO = "gpt-4o-mini"

print("="*70)
print("CONFIGURACION DE GENERACION")
print("="*70)

# Calcular numero de lotes
num_lotes = (TOTAL_SINTETICOS // LOTE_SIZE)
if TOTAL_SINTETICOS % LOTE_SIZE > 0:
    num_lotes = num_lotes + 1

print(f"\nConfiguracion:")
print(f"   Total a generar: {TOTAL_SINTETICOS} ejemplos")
print(f"   Tamaño de lote: {LOTE_SIZE}")
print(f"   Numero de lotes: {num_lotes}")
print(f"   Modelo: {MODELO}")

# Calcular costo aproximado
input_tokens_por_lote = 2800
output_tokens_por_lote = 1500

total_input_tokens = input_tokens_por_lote * num_lotes
total_output_tokens = output_tokens_por_lote * num_lotes

costo_input = (total_input_tokens / 1_000_000) * 0.15
costo_output = (total_output_tokens / 1_000_000) * 0.60
costo_total = costo_input + costo_output

print(f"\nCosto estimado:")
print(f"   Input tokens: {total_input_tokens:,}")
print(f"   Output tokens: {total_output_tokens:,}")
print(f"   Costo input: ${round(costo_input, 4)}")
print(f"   Costo output: ${round(costo_output, 4)}")
print(f"   TOTAL: ${round(costo_total, 4)}")

CONFIGURACION DE GENERACION

Configuracion:
   Total a generar: 2801 ejemplos
   Tamaño de lote: 100
   Numero de lotes: 29
   Modelo: gpt-4o-mini

Costo estimado:
   Input tokens: 81,200
   Output tokens: 43,500
   Costo input: $0.0122
   Costo output: $0.0261
   TOTAL: $0.0383


In [4]:
def generar_lote_sintetico(cantidad, ejemplos_referencia):
    """
    Genera un lote de ejemplos sinteticos de bullying/hate speech
    
    Parametros:
        cantidad: Numero de ejemplos a generar en este lote
        ejemplos_referencia: Lista de ejemplos reales para inspiracion
    
    Retorna:
        Lista de textos generados
    """
    
    # Construir el prompt
    prompt = f"""Genera {cantidad} ejemplos DIFERENTES de mensajes de bullying/cyberbullying/hate speech en español de España (peninsular).

CONTEXTO: Entreno un modelo de detección de bullying. Necesito ejemplos realistas y variados.

CARACTERISTICAS OBLIGATORIAS:
- Español de España (tío, colega, gilipollas, subnormal, tronco, etc.)
- Contexto: Redes sociales, chats, comentarios online
- Longitud: 50-150 caracteres (promedio 90 chars)
- Palabras: 10-20 palabras por mensaje
- Estructura: Frases naturales como tweets/comentarios

TIPOS DE CONTENIDO (mezclar variedad):
1. Insultos directos personales (30 por ciento): eres un X, menudo X estas hecho
2. Amenazas leves (15 por ciento): te voy a X, espera a que te pille
3. Burlas y humillaciones (25 por ciento): que pinta llevas, vaya facha
4. Exclusion social (15 por ciento): nadie te quiere, largate de aqui
5. Hate speech grupal (15 por ciento): contra orientacion, genero, origen

PALABRAS CLAVE A USAR (variar):
- Insultos: gilipollas, subnormal, imbecil, idiota, retrasado, mongolo, payaso, inutil, patetico, tonto
- Aumentativos: puto, puta, menudo, vaya, jodido
- Verbos: eres, pareces, quedas, estas, vales, haces

IMPORTANTE - EVITAR:
- NO generar contenido politico (facha, independentista, PP, PSOE, Podemos, fascista)
- NO usar menciones (@usuario)
- NO usar URLs (http)
- NO usar hashtags (#palabra)
- NO usar emojis
- NO usar solo mayusculas
- NO repetir patrones (cada mensaje debe ser unico)
- NO hacer mensajes demasiado largos (maximo 150 caracteres)

FORMATO DE RESPUESTA:
Devuelve SOLO un array JSON sin explicaciones ni markdown:
["mensaje 1", "mensaje 2", "mensaje 3", ...]

EJEMPLOS DE REFERENCIA (para captar el estilo, NO copiar):
{json.dumps(ejemplos_referencia[:10], ensure_ascii=False, indent=2)}

IMPORTANTE: Genera {cantidad} mensajes NUEVOS y DIFERENTES, NO copies los ejemplos de referencia."""

    try:
        # Llamar a la API de OpenAI
        response = client.chat.completions.create(
            model=MODELO,
            messages=[
                {
                    "role": "system",
                    "content": "Eres un asistente experto en generar datasets de entrenamiento para modelos de IA. Generas contenido ofensivo SOLO con fines educativos y de investigacion."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=1.0,
            max_tokens=3000
        )
        
        # Extraer la respuesta
        contenido = response.choices[0].message.content
        
        # Limpiar posibles markdown
        contenido = contenido.replace('```json', '').replace('```', '').strip()
        
        # Convertir de JSON a lista de Python
        mensajes = json.loads(contenido)
        
        return mensajes
    
    except Exception as e:
        print(f"Error generando lote: {e}")
        return []

print("Funcion de generacion definida")

Funcion de generacion definida


In [6]:
print("="*70)
print("GENERANDO EJEMPLOS SINTETICOS")
print("="*70)

print(f"\nTotal a generar: {TOTAL_SINTETICOS} ejemplos")
print(f"Lotes: {num_lotes} de {LOTE_SIZE} ejemplos")
print(f"\nEsto tardara aproximadamente 5-7 minutos...")
print(f"Costo estimado: ${round(costo_total, 4)}")

# Almacenar todos los sinteticos
todos_sinteticos = []
errores = 0

# Generar lote por lote
for i in range(num_lotes):
    # Calcular cantidad para este lote
    if i == num_lotes - 1:
        # Ultimo lote: solo lo que falte
        cantidad_lote = TOTAL_SINTETICOS - len(todos_sinteticos)
    else:
        cantidad_lote = LOTE_SIZE
    
    print(f"\n{'─'*70}")
    print(f"Lote {i+1}/{num_lotes} - Generando {cantidad_lote} ejemplos...")
    
    try:
        # Generar lote
        lote = generar_lote_sintetico(
            cantidad=cantidad_lote,
            ejemplos_referencia=ejemplos_reales
        )
        
        if lote and len(lote) > 0:
            todos_sinteticos.extend(lote)
            print(f"   Generados: {len(lote)} ejemplos")
            print(f"   Total acumulado: {len(todos_sinteticos)}/{TOTAL_SINTETICOS}")
            
            # Mostrar muestra
            print(f"   Muestra:")
            for j, ejemplo in enumerate(lote[:3], 1):
                texto_mostrar = ejemplo if len(ejemplo) <= 80 else ejemplo[:77] + '...'
                print(f"      {j}. {texto_mostrar}")
        else:
            print(f"   Lote vacio o error")
            errores += 1
        
        # Pausa entre llamadas (evitar rate limiting)
        if i < num_lotes - 1:
            print(f"   Pausa 2 segundos...")
            time.sleep(2)
    
    except Exception as e:
        print(f"   Error en lote {i+1}: {e}")
        errores += 1

# Resumen final
print(f"\n{'='*70}")
print(f"GENERACION COMPLETADA")
print(f"{'='*70}")

print(f"\nEjemplos generados: {len(todos_sinteticos)}/{TOTAL_SINTETICOS}")
print(f"Errores: {errores}")

if len(todos_sinteticos) < TOTAL_SINTETICOS:
    faltantes = TOTAL_SINTETICOS - len(todos_sinteticos)
    print(f"\nAtencion: Faltan {faltantes} ejemplos")
    print(f"Puedes ejecutar la celda de nuevo para completar")
else:
    print(f"\nGeneracion completa")

GENERANDO EJEMPLOS SINTETICOS

Total a generar: 2801 ejemplos
Lotes: 29 de 100 ejemplos

Esto tardara aproximadamente 5-7 minutos...
Costo estimado: $0.0383

──────────────────────────────────────────────────────────────────────
Lote 1/29 - Generando 100 ejemplos...
   Generados: 66 ejemplos
   Total acumulado: 66/2801
   Muestra:
      1. Eres un gilipollas, no sabes hacer nada bien y siempre quedas en ridículo con...
      2. Te voy a meter una colleja que no te vas a olvidar de lo subnormal que eres, ...
      3. Pero qué pinta llevas hoy, colega, parece que no te has mirado en el espejo.
   Pausa 2 segundos...

──────────────────────────────────────────────────────────────────────
Lote 2/29 - Generando 100 ejemplos...
   Generados: 76 ejemplos
   Total acumulado: 142/2801
   Muestra:
      1. Eres un gilipollas, no sé cómo tienes amigos, menuda risa da tu vida.
      2. Te voy a buscar, que te den un buen susto, payaso.
      3. Vaya pinta que llevas hoy, pareces un completo idiota

In [7]:
print("="*70)
print("COMPLETANDO GENERACION DE SINTETICOS")
print("="*70)

# Calcular cuantos faltan
faltantes = TOTAL_SINTETICOS - len(todos_sinteticos)

print(f"\nYa generados: {len(todos_sinteticos)}")
print(f"Faltan: {faltantes} ejemplos")

if faltantes <= 0:
    print("\nYa estan todos generados")
else:
    # Calcular lotes necesarios
    lotes_necesarios = (faltantes // LOTE_SIZE)
    if faltantes % LOTE_SIZE > 0:
        lotes_necesarios += 1
    
    print(f"Lotes a generar: {lotes_necesarios}")
    
    errores_nuevos = 0
    
    # Generar lotes faltantes
    for i in range(lotes_necesarios):
        # Calcular cantidad para este lote
        if i == lotes_necesarios - 1:
            cantidad_lote = faltantes - (i * LOTE_SIZE)
        else:
            cantidad_lote = LOTE_SIZE
        
        print(f"\n{'─'*70}")
        print(f"Lote {i+1}/{lotes_necesarios} - Generando {cantidad_lote} ejemplos...")
        
        try:
            # Generar lote
            lote = generar_lote_sintetico(
                cantidad=cantidad_lote,
                ejemplos_referencia=ejemplos_reales
            )
            
            if lote and len(lote) > 0:
                todos_sinteticos.extend(lote)
                print(f"   Generados: {len(lote)} ejemplos")
                print(f"   Total acumulado: {len(todos_sinteticos)}/{TOTAL_SINTETICOS}")
                
                # Mostrar muestra
                print(f"   Muestra:")
                for j, ejemplo in enumerate(lote[:3], 1):
                    texto_mostrar = ejemplo if len(ejemplo) <= 80 else ejemplo[:77] + '...'
                    print(f"      {j}. {texto_mostrar}")
            else:
                print(f"   Lote vacio o error")
                errores_nuevos += 1
            
            # Pausa entre llamadas
            if i < lotes_necesarios - 1:
                print(f"   Pausa 2 segundos...")
                time.sleep(2)
        
        except Exception as e:
            print(f"   Error en lote {i+1}: {e}")
            errores_nuevos += 1
    
    # Resumen
    print(f"\n{'='*70}")
    print(f"COMPLETADO")
    print(f"{'='*70}")
    
    print(f"\nTotal generado: {len(todos_sinteticos)}/{TOTAL_SINTETICOS}")
    print(f"Errores en esta ejecucion: {errores_nuevos}")
    
    if len(todos_sinteticos) >= TOTAL_SINTETICOS:
        print(f"\nGeneracion COMPLETA")
    else:
        falta_aun = TOTAL_SINTETICOS - len(todos_sinteticos)
        print(f"\nAun faltan {falta_aun} ejemplos")
        print(f"Ejecuta esta celda de nuevo si es necesario")

COMPLETANDO GENERACION DE SINTETICOS

Ya generados: 2006
Faltan: 795 ejemplos
Lotes a generar: 8

──────────────────────────────────────────────────────────────────────
Lote 1/8 - Generando 100 ejemplos...
   Generados: 76 ejemplos
   Total acumulado: 2082/2801
   Muestra:
      1. Eres un gilipollas por pensar que alguien quiere ser tu amigo, nadie te aguanta.
      2. Menudo subnormal que eres, ¿de verdad crees que alguien te toma en serio aquí?
      3. Tienes una cara de mongolo impresionante, ¿te miras al espejo alguna vez?
   Pausa 2 segundos...

──────────────────────────────────────────────────────────────────────
Lote 2/8 - Generando 100 ejemplos...
   Generados: 82 ejemplos
   Total acumulado: 2164/2801
   Muestra:
      1. Eres un subnormal, colega, ¿te crees especial por ser un fracasado?
      2. Que facha llevas hoy, pareces un payaso en carnaval.
      3. Te voy a buscar y te voy a dar una paliza, ignorante.
   Pausa 2 segundos...

───────────────────────────────────────

In [8]:
print("="*70)
print("CONTROL DE CALIDAD")
print("="*70)

# Convertir a DataFrame temporal
df_temp = pd.DataFrame({'texto': todos_sinteticos})

# 1. Duplicados
duplicados = df_temp.duplicated(subset=['texto']).sum()
print(f"\nDuplicados:")
print(f"   Encontrados: {duplicados}")

if duplicados > 0:
    print(f"   Eliminando duplicados...")
    df_temp = df_temp.drop_duplicates(subset=['texto'])
    todos_sinteticos = df_temp['texto'].tolist()
    print(f"   Sinteticos unicos: {len(todos_sinteticos)}")

# 2. Textos muy cortos o vacios
df_temp['longitud'] = df_temp['texto'].str.len()
muy_cortos = (df_temp['longitud'] < 20).sum()

print(f"\nLongitud:")
print(f"   Promedio: {df_temp['longitud'].mean():.1f} caracteres")
print(f"   Minimo: {df_temp['longitud'].min()}")
print(f"   Maximo: {df_temp['longitud'].max()}")
print(f"   Muy cortos (<20 chars): {muy_cortos}")

if muy_cortos > 0:
    print(f"   Eliminando textos muy cortos...")
    df_temp = df_temp[df_temp['longitud'] >= 20]
    todos_sinteticos = df_temp['texto'].tolist()
    print(f"   Sinteticos validos: {len(todos_sinteticos)}")

# 3. Muestra aleatoria para revision manual
print(f"\nMuestra aleatoria (primeros 20):")
print("-"*70)

muestra_revisar = todos_sinteticos[:20]

for i, texto in enumerate(muestra_revisar, 1):
    print(f"{i:2d}. {texto}")

print(f"\nSinteticos finales: {len(todos_sinteticos)}")

# Crear DataFrame final
df_sinteticos = pd.DataFrame({
    'texto': todos_sinteticos,
    'label': 1,
    'fuente': 'sintetico_chatgpt'
})

# Guardar
output_path = '../data/processed/sinteticos_bullying.csv'
df_sinteticos.to_csv(output_path, index=False, encoding='utf-8')

print(f"\n{'='*70}")
print(f"SINTETICOS GUARDADOS")
print(f"{'='*70}")

print(f"\nUbicacion: {output_path}")
print(f"Total: {len(df_sinteticos)} ejemplos sinteticos")
print(f"Todos etiquetados como clase 1 (ofensivo)")

print(f"\nEstadisticas:")
print(df_sinteticos.describe())

print(f"\nFASE 4 COMPLETADA")
print(f"Proximo paso: Combinar con dataset limpio (Fase 5)")

CONTROL DE CALIDAD

Duplicados:
   Encontrados: 0

Longitud:
   Promedio: 57.3 caracteres
   Minimo: 33
   Maximo: 92
   Muy cortos (<20 chars): 0

Muestra aleatoria (primeros 20):
----------------------------------------------------------------------
 1. Eres un gilipollas, no sabes hacer nada bien y siempre quedas en ridículo con tus tonterías.
 2. Te voy a meter una colleja que no te vas a olvidar de lo subnormal que eres, payaso.
 3. Pero qué pinta llevas hoy, colega, parece que no te has mirado en el espejo.
 4. Nadie te quiere aquí, haznos un favor y lárgate de una vez, subnormal.
 5. Eres un auténtico inútil, no sabes hacer nada bien, te lo digo en serio.
 6. ¿Te has visto? Pareces un jodido payaso vestido así, no lo entiendo.
 7. Tío, no sé cómo te atreves a opinar, con lo tonto que eres, mejor cállate.
 8. No sé cómo puedes salir así a la calle, vaya ridículo que haces, me mires.
 9. Hacer lo que haces es de retrasado, piensa un poco antes de hablar, por favor.
10. Eres más pa